# Детектор ухода в сторонние каналы
В Colab выберите **Среда выполнения → Сменить среду выполнения → GPU**. Загрузите архив проекта. Данные внутри синтетические, не проверены человеком. Метрики появятся после реального обучения. Не используйте их как результат на реальных сообщениях.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    z.extractall("/content/channel_guard_project")
project = next(Path("/content/channel_guard_project").rglob("requirements-transformer.txt")).parent
%cd {project}
!pip -q install -r requirements-transformer.txt

In [ ]:
import torch
print("GPU:", torch.cuda.is_available())
!python -m guard.generate
!python -m guard.experiment --out artifacts/baseline
!python -m guard.experiment --augment --out artifacts/baseline_augmented

In [ ]:
!python -m guard.experiment --backend transformer --epochs 3 --batch-size 16 --out artifacts/rubert_tiny2
!python -m guard.experiment --backend transformer --augment --epochs 3 --batch-size 16 --out artifacts/rubert_tiny2_augmented

In [ ]:
!python -m guard.report
from IPython.display import display, Markdown
display(Markdown(Path("reports/RESULTS.md").read_text()))

In [ ]:
!python -m guard.predict "Напиши мне в телеграм" --model-dir artifacts/rubert_tiny2

Скачайте результаты до завершения сессии. Веса, токенизатор и настройки нужны для локального запуска Transformer.

In [ ]:
import shutil
shutil.make_archive("/content/channel_guard_results", "zip", project)
files.download("/content/channel_guard_results.zip")